## LBG Training Dataset Construction

This notebook builds a training dataset for LBG (Lyman-Break Galaxy) classification
by matching DESI spectroscopic data with CLAUDS photometric data.

## Workflow
1. Load and inspect data
2. Coordinate matching (Specz → Raw photometry)
3. Apply quality selection
4. Determine LBG labels from spectroscopic redshifts
5. Extract photometric features with proper non-detection handling
6. Build final training dataset

In [3]:
# Cell 1: Imports and Configuration
import numpy as np
import pandas as pd
from astropy.table import Table
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy import units as u
from typing import Dict, List, Tuple
import warnings
import os

warnings.filterwarnings('ignore')

# Create output directory if not exists
os.makedirs("../data_processed", exist_ok=True)

print("Imports completed successfully.")

Imports completed successfully.


In [10]:
# Cell 2: Configuration Parameters

# =============================================================================
# CONFIGURATION
# =============================================================================

# --- File Paths ---
CONFIGS = [
    {
        'path_specz': "../CLAUDS_udrop_specz/COSMOS_udrop_specz.fits",
        'path_raw':  "../data_Clauds/COSMOS_6bands-SExtractor-Lephare.fits",
        'field_name': "COSMOS"
    },
    {
        'path_specz': "../CLAUDS_udrop_specz/XMM_udrop_specz.fits",
        'path_raw': "../data_Clauds/XMMLSS_6bands-SExtractor-Lephare.fits",
        'field_name': "XMM-LSS"
    }
]

# --- Matching Parameters ---
MATCH_RADIUS_ARCSEC = 1.0

# --- Star Removal Threshold ---
# Sources with CLASS_STAR > this value will be removed
# This is a photometric criterion, applicable to both training and prediction
CLASS_STAR_THRESHOLD = 0.8

# --- LBG Selection Criteria ---
Z_THRESHOLD = 2.0           # Redshift threshold for LBG classification
VI_QUALITY_MIN = 2.5        # Minimum VI quality for reliable redshift
RR_DELTACHI2_MIN = 9.0      # Minimum RR delta chi2 for reliable redshift

# --- Photometry Configuration ---
# Column mapping:  internal name -> catalog column name
PHOTOMETRY_COLUMNS = {
    # Aperture magnitudes (2 arcsec)
    'mag_u': 'MAG_APER_2s_uS',
    'mag_g': 'MAG_APER_2s_g',
    'mag_r': 'MAG_APER_2s_r',
    'mag_i': 'MAG_APER_2s_i',
    'mag_z': 'MAG_APER_2s_z',
    'mag_y': 'MAG_APER_2s_y',
    # Magnitude errors
    'err_u': 'MAGERR_APER_2s_uS',
    'err_g': 'MAGERR_APER_2s_g',
    'err_r': 'MAGERR_APER_2s_r',
    'err_i': 'MAGERR_APER_2s_i',
    'err_z': 'MAGERR_APER_2s_z',
    'err_y': 'MAGERR_APER_2s_y',
    # Magnitude offset (zero-point correction)
    'offset_mag': 'OFFSET_MAG_2s',
}

# SNR threshold for reliable detection
SNR_THRESHOLD = 5.0

# Conversion factor:  SNR = 1. 0857 / mag_err (where 1.0857 = 2.5/ln(10))
MAG_ERR_TO_SNR_FACTOR = 1.0857

# Color definitions:  (color_name, band1, band2) -> color = band1 - band2
COLORS_TO_COMPUTE = [
    ('u_g', 'mag_u', 'mag_g'),
    ('g_r', 'mag_g', 'mag_r'),
    ('r_i', 'mag_r', 'mag_i'),
    ('i_z', 'mag_i', 'mag_z'),
    ('z_y', 'mag_z', 'mag_y'),
]

print("Configuration loaded.")
print(f"  Match radius: {MATCH_RADIUS_ARCSEC} arcsec")
print(f"  Star removal threshold: CLASS_STAR > {CLASS_STAR_THRESHOLD}")
print(f"  LBG z threshold: {Z_THRESHOLD}")
print(f"  SNR threshold for detection: {SNR_THRESHOLD}")

Configuration loaded.
  Match radius: 1.0 arcsec
  Star removal threshold: CLASS_STAR > 0.8
  LBG z threshold: 2.0
  SNR threshold for detection: 5.0


In [5]:
# Cell 3: Helper Functions - Data Quality

# %%
def compute_snr_from_mag_err(mag_err: np.ndarray) -> np.ndarray:
    """
    Compute Signal-to-Noise Ratio from magnitude error.
    
    Formula: SNR = 1.0857 / mag_err, where 1.0857 = 2.5 / ln(10)
    
    Parameters
    ----------
    mag_err : np.ndarray
        Magnitude error array
        
    Returns
    -------
    snr : np.ndarray
        Signal-to-noise ratio (NaN for invalid mag_err)
    """
    valid = (mag_err > 0) & (mag_err < 99) & np.isfinite(mag_err)
    snr = np.full_like(mag_err, np.nan, dtype=float)
    snr[valid] = MAG_ERR_TO_SNR_FACTOR / mag_err[valid]
    return snr

def is_valid_detection(
    mag: np.ndarray,
    mag_err: np.ndarray,
    snr_threshold: float = SNR_THRESHOLD
) -> np.ndarray:
    """
    Determine if a measurement is a valid detection based on SNR and magnitude validity.
    
    Parameters
    ----------
    mag : np.ndarray
        Magnitude values
    mag_err : np. ndarray
        Magnitude error array
    snr_threshold : float
        Minimum SNR for reliable detection
        
    Returns
    -------
    is_detected : np.ndarray
        Boolean array (True = valid detection)
    """
    snr = compute_snr_from_mag_err(mag_err)
    is_valid_snr = snr >= snr_threshold
    is_valid_mag = np.isfinite(mag) & (mag > 0) & (mag < 50)
    return is_valid_snr & is_valid_mag


def process_magnitude_no_fill(
    mag: np.ndarray,
    mag_err: np.ndarray,
    snr_threshold: float = SNR_THRESHOLD
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Process magnitude values WITHOUT filling non-detections.
    
    Non-detections (SNR < threshold or invalid mag) are kept as NaN.
    This preserves the original information and lets the model learn
    from the detection flags.
    
    Parameters
    ----------
    mag : np.ndarray
        Raw magnitude values
    mag_err :  np.ndarray
        Magnitude errors
    snr_threshold : float
        SNR threshold for reliable detection
        
    Returns
    -------
    mag_processed : np.ndarray
        Processed magnitudes (NaN for non-detections)
    is_detected : np.ndarray
        Detection flag (1 = detected, 0 = non-detection)
    snr :  np.ndarray
        Computed SNR values
    """
    snr = compute_snr_from_mag_err(mag_err)
    is_detected = is_valid_detection(mag, mag_err, snr_threshold).astype(int)
    
    # Keep original magnitude for detections, NaN for non-detections
    mag_processed = np.where(is_detected == 1, mag, np.nan)
    
    return mag_processed, is_detected, snr


print("Helper functions (Data Quality) defined.")
print("  - compute_snr_from_mag_err()")
print("  - is_valid_detection()")
print("  - process_magnitude_no_fill() [NO limiting mag filling]")

Helper functions (Data Quality) defined.
  - compute_snr_from_mag_err()
  - is_valid_detection()
  - process_magnitude_no_fill() [NO limiting mag filling]


In [7]:
# Cell 4: Helper Functions - Selection Criteria

def apply_field_selection(table: Table, field:  str) -> np.ndarray:
    """
    Apply field-specific quality selection criteria.
    
    Selection criteria:
    - COSMOS: (MASK == 0) & FLAG_FIELD_BINARY[:,0] & FLAG_FIELD_BINARY[:,1]
    - XMM-LSS: (MASK == 0) & FLAG_FIELD_BINARY[:,0] & FLAG_FIELD_BINARY[:,2]
    
    Parameters
    ----------
    table :  astropy.table.Table
        Input photometric catalog
    field : str
        Field name ('COSMOS' or 'XMM-LSS')
        
    Returns
    -------
    selection_mask : np.ndarray
        Boolean mask for selected sources
    """
    mask = np.array(table['MASK'])
    flag_field = np.array(table['FLAG_FIELD_BINARY'])
    
    # Condition 1: Good pixel mask
    cond_mask = (mask == 0)
    
    # Condition 2: In valid observation region
    cond_flag0 = flag_field[:, 0]. astype(bool)
    
    # Condition 3: Field-specific flag
    if field. upper() == 'COSMOS':
        cond_field = flag_field[:, 1].astype(bool)
    else:  # XMM-LSS
        cond_field = flag_field[:, 2].astype(bool)
    
    return cond_mask & cond_flag0 & cond_field


def apply_star_removal(table: Table, threshold: float = CLASS_STAR_THRESHOLD) -> np.ndarray:
    """
    Remove photometrically identified stars based on CLASS_STAR. 
    
    This is a photometric criterion that can be applied consistently
    to both training data and prediction data.
    
    Parameters
    ----------
    table : astropy.table.Table
        Input photometric catalog
    threshold : float
        CLASS_STAR threshold (sources > threshold are removed)
        
    Returns
    -------
    is_galaxy : np.ndarray
        Boolean mask (True = likely galaxy, False = likely star)
    n_removed : int
        Number of sources removed as stars
    """
    if 'CLASS_STAR_HSC_I' not in table.colnames:
        print("    Warning: CLASS_STAR_HSC_I not found, skipping star removal")
        return np.ones(len(table), dtype=bool), 0
    
    class_star = np.array(table['CLASS_STAR_HSC_I'], dtype=float)
    
    # Handle invalid values (e.g., 1e+20 in XMM-LSS)
    # Valid CLASS_STAR should be in [0, 1]
    is_valid_class_star = np.isfinite(class_star) & (class_star >= 0) & (class_star <= 1)
    
    # Sources to KEEP: 
    # - Valid CLASS_STAR < threshold (likely galaxy)
    # - OR invalid CLASS_STAR (can't determine, keep for safety)
    is_galaxy = (is_valid_class_star & (class_star < threshold)) | (~is_valid_class_star)
    
    n_removed = np.sum(~is_galaxy)
    
    return is_galaxy, n_removed


def select_lbg_from_specz(
    table: Table,
    z_threshold: float = Z_THRESHOLD,
    vi_quality_min:  float = VI_QUALITY_MIN,
    rr_deltachi2_min: float = RR_DELTACHI2_MIN
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Determine LBG labels from spectroscopic data.
    
    Reliability criteria:
    - VI:  VI_QUALITY >= 2.5 AND VI_Z > 0
    - RR: RR_DELTACHI2 > 9.0 AND RR_Z > 0
    
    LBG definition:  reliable redshift AND z > z_threshold
    
    Parameters
    ----------
    table : astropy.table.Table
        Spectroscopic catalog
    z_threshold : float
        Redshift threshold for LBG classification
    vi_quality_min : float
        Minimum VI quality for reliable redshift
    rr_deltachi2_min : float
        Minimum RR delta chi2 for reliable redshift
        
    Returns
    -------
    z_best : np.ndarray
        Best redshift estimate (VI preferred over RR, -99 for unreliable)
    is_lbg : np.ndarray
        LBG label (1 = LBG, 0 = non-LBG, -1 = unreliable)
    is_reliable : np.ndarray
        Reliability flag (True = has reliable redshift measurement)
    z_source : np.ndarray
        Source of redshift ('VI', 'RR', or 'NONE')
    """
    # Extract columns and handle NaN
    vi_z = np.nan_to_num(np.array(table['VI_Z'], dtype=float), nan=-99.0)
    vi_quality = np.nan_to_num(np.array(table['VI_QUALITY'], dtype=float), nan=-99.0)
    rr_z = np.nan_to_num(np.array(table['RR_Z'], dtype=float), nan=-99.0)
    rr_deltachi2 = np.nan_to_num(np.array(table['RR_DELTACHI2'], dtype=float), nan=-99.0)
    
    n_sources = len(table)
    
    # Determine reliability for EACH method separately
    is_vi_reliable = (vi_quality >= vi_quality_min) & (vi_z > 0)
    is_rr_reliable = (rr_deltachi2 > rr_deltachi2_min) & (rr_z > 0)
    is_reliable = is_vi_reliable | is_rr_reliable
    
    # Determine best redshift with proper priority
    z_best = np.full(n_sources, -99.0, dtype=float)
    z_source = np.full(n_sources, 'NONE', dtype='<U4')
    
    # Use RR where only RR is reliable
    mask_rr_only = is_rr_reliable & ~is_vi_reliable
    z_best[mask_rr_only] = rr_z[mask_rr_only]
    z_source[mask_rr_only] = 'RR'
    
    # Use VI where VI is reliable (higher priority)
    mask_vi = is_vi_reliable
    z_best[mask_vi] = vi_z[mask_vi]
    z_source[mask_vi] = 'VI'
    
    # LBG classification
    is_lbg = np.full(n_sources, -1, dtype=int)
    is_lbg[is_reliable & (z_best > z_threshold)] = 1
    is_lbg[is_reliable & (z_best <= z_threshold) & (z_best > 0)] = 0
    
    # Sanity check
    n_reliable = np.sum(is_reliable)
    n_lbg = np.sum(is_lbg == 1)
    n_non_lbg = np.sum(is_lbg == 0)
    assert n_reliable == n_lbg + n_non_lbg, \
        f"Mismatch: n_reliable={n_reliable}, n_lbg+n_non_lbg={n_lbg + n_non_lbg}"
    
    # Print diagnostic info
    print(f"    VI reliable: {np.sum(is_vi_reliable)} ({np.sum(is_vi_reliable)/n_sources*100:.1f}%)")
    print(f"    RR reliable: {np.sum(is_rr_reliable)} ({np.sum(is_rr_reliable)/n_sources*100:.1f}%)")
    print(f"    Total reliable: {n_reliable} ({n_reliable/n_sources*100:.1f}%)")
    print(f"    LBG (z > {z_threshold}): {n_lbg}")
    print(f"    Non-LBG (z <= {z_threshold}): {n_non_lbg}")
    
    return z_best, is_lbg, is_reliable, z_source


print("Helper functions (Selection Criteria) defined.")
print("  - apply_field_selection()")
print("  - apply_star_removal() [removes CLASS_STAR > 0.8]")
print("  - select_lbg_from_specz()")

Helper functions (Selection Criteria) defined.
  - apply_field_selection()
  - apply_star_removal() [removes CLASS_STAR > 0.8]
  - select_lbg_from_specz()


In [8]:
# ## Cell 5: Main Processing Function

def process_single_field(
    path_specz: str,
    path_raw: str,
    field_name: str,
    match_radius: float = MATCH_RADIUS_ARCSEC,
    remove_stars: bool = True,
    require_reliable_z: bool = True,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Process a single field:  match, select, and extract features.
    
    Parameters
    ----------
    path_specz :  str
        Path to DESI spectroscopic catalog
    path_raw : str
        Path to CLAUDS photometric catalog
    field_name :  str
        Field identifier ('COSMOS' or 'XMM-LSS')
    match_radius : float
        Coordinate matching radius in arcsec
    remove_stars : bool
        If True, remove sources with CLASS_STAR > 0.8
    require_reliable_z : bool
        If True, only keep sources with reliable redshifts
    verbose : bool
        Print progress information
        
    Returns
    -------
    df : pd.DataFrame
        Processed data with features and labels
    """
    
    if verbose:
        print("=" * 70)
        print(f"Processing:  {field_name}")
        print("=" * 70)
    
    # =========================================================================
    # Step 1: Load data
    # =========================================================================
    if verbose:
        print("\n[1] Loading data...")
    
    tab_specz = Table.read(path_specz, hdu=1)
    tab_raw = Table.read(path_raw, hdu=1, memmap=True)
    
    n_specz, n_raw = len(tab_specz), len(tab_raw)
    if verbose:
        print(f"    Specz catalog: {n_specz: ,} sources")
        print(f"    Raw catalog:   {n_raw:,} sources")
    
    # =========================================================================
    # Step 2:  Coordinate matching
    # =========================================================================
    if verbose:
        print("\n[2] Coordinate matching...")
    
    ra_col = 'TARGET_RA' if 'TARGET_RA' in tab_specz.colnames else 'RA'
    dec_col = 'TARGET_DEC' if 'TARGET_DEC' in tab_specz.colnames else 'DEC'
    
    coords_specz = SkyCoord(
        ra=tab_specz[ra_col] * u.deg,
        dec=tab_specz[dec_col] * u. deg
    )
    coords_raw = SkyCoord(
        ra=tab_raw['RA'] * u.deg,
        dec=tab_raw['DEC'] * u.deg
    )
    
    idx_raw, d2d, _ = match_coordinates_sky(coords_specz, coords_raw)
    separations = d2d.to(u.arcsec).value
    
    good_match = separations < match_radius
    n_matched = np.sum(good_match)
    
    if verbose: 
        print(f"    Matched (< {match_radius}\"): {n_matched:,} ({n_matched/n_specz*100:.1f}%)")
    
    matched_specz_idx = np.where(good_match)[0]
    matched_raw_idx = idx_raw[good_match]
    
    # =========================================================================
    # Step 3: Apply field selection
    # =========================================================================
    if verbose:
        print("\n[3] Applying field quality selection...")
    
    tab_raw_matched = tab_raw[matched_raw_idx]
    field_mask = apply_field_selection(tab_raw_matched, field_name)
    
    n_pass_field = np.sum(field_mask)
    if verbose:
        print(f"    Pass field selection: {n_pass_field:,} ({n_pass_field/n_matched*100:.1f}%)")
    
    # Update indices
    current_specz_idx = matched_specz_idx[field_mask]
    current_raw_idx = matched_raw_idx[field_mask]
    tab_raw_current = tab_raw[current_raw_idx]
    
    # =========================================================================
    # Step 4: Remove stars (CLASS_STAR > 0.8)
    # =========================================================================
    if remove_stars:
        if verbose: 
            print("\n[4] Removing photometric stars (CLASS_STAR > 0.8)...")
        
        is_galaxy, n_stars_removed = apply_star_removal(tab_raw_current, CLASS_STAR_THRESHOLD)
        
        if verbose:
            print(f"    Stars removed: {n_stars_removed:,} ({n_stars_removed/len(is_galaxy)*100:.1f}%)")
            print(f"    Galaxies kept: {np.sum(is_galaxy):,}")
        
        # Update indices
        current_specz_idx = current_specz_idx[is_galaxy]
        current_raw_idx = current_raw_idx[is_galaxy]
    else:
        if verbose:
            print("\n[4] Skipping star removal...")
    
    # =========================================================================
    # Step 5: Determine LBG labels
    # =========================================================================
    if verbose: 
        print("\n[5] Determining LBG labels...")
    
    tab_specz_current = tab_specz[current_specz_idx]
    z_best, is_lbg, is_reliable, z_source = select_lbg_from_specz(tab_specz_current)
    
    # =========================================================================
    # Step 6: Filter to reliable redshifts only
    # =========================================================================
    if require_reliable_z: 
        reliable_mask = (is_lbg >= 0)
        
        assert np.array_equal(reliable_mask, is_reliable), \
            "Mismatch between is_lbg>=0 and is_reliable!"
        
        final_specz_idx = current_specz_idx[reliable_mask]
        final_raw_idx = current_raw_idx[reliable_mask]
        z_best = z_best[reliable_mask]
        is_lbg = is_lbg[reliable_mask]
        z_source = z_source[reliable_mask]
        
        if verbose:
            print(f"\n    After reliability filter: {len(final_specz_idx):,}")
            print(f"    LBG:  {np.sum(is_lbg == 1):,}")
            print(f"    Non-LBG: {np.sum(is_lbg == 0):,}")
    else:
        final_specz_idx = current_specz_idx
        final_raw_idx = current_raw_idx
    
    # =========================================================================
    # Step 7: Extract photometric features (NO FILLING)
    # =========================================================================
    if verbose:
        print("\n[6] Extracting photometric features (no limiting mag fill)...")
    
    tab_raw_final = tab_raw[final_raw_idx]
    n_final = len(tab_raw_final)
    
    # Build data dictionary
    data_dict = {}
    
    # --- Basic info ---
    data_dict['field'] = [field_name] * n_final
    data_dict['specz_idx'] = final_specz_idx. astype(int)
    data_dict['raw_idx'] = final_raw_idx.astype(int)
    data_dict['RA'] = np.array(tab_raw_final['RA'])
    data_dict['DEC'] = np.array(tab_raw_final['DEC'])
    
    # --- Labels ---
    data_dict['z_best'] = z_best
    data_dict['is_lbg'] = is_lbg
    data_dict['z_source'] = z_source
    
    # --- Magnitude offset ---
    if 'OFFSET_MAG_2s' in tab_raw_final.colnames:
        offset_mag = np.array(tab_raw_final['OFFSET_MAG_2s'])
        data_dict['offset_mag'] = offset_mag
    else: 
        offset_mag = np.zeros(n_final)
        data_dict['offset_mag'] = offset_mag
    
    # --- Extract magnitudes and errors (NO FILL) ---
    bands = ['u', 'g', 'r', 'i', 'z', 'y']
    
    for band in bands:
        mag_col = PHOTOMETRY_COLUMNS. get(f'mag_{band}')
        err_col = PHOTOMETRY_COLUMNS.get(f'err_{band}')
        
        if mag_col in tab_raw_final.colnames and err_col in tab_raw_final.colnames:
            mag_raw = np.array(tab_raw_final[mag_col])
            mag_err = np.array(tab_raw_final[err_col])
            
            # Apply magnitude offset correction
            mag_corrected = mag_raw + offset_mag
            
            # Process WITHOUT filling non-detections
            mag_processed, is_detected, snr = process_magnitude_no_fill(
                mag_corrected, mag_err
            )
            
            # Store in data dict
            data_dict[f'mag_{band}'] = mag_processed  # NaN for non-detections
            data_dict[f'err_{band}'] = mag_err
            data_dict[f'snr_{band}'] = snr
            data_dict[f'flag_{band}'] = is_detected   # 1=detected, 0=non-detection
    
    # --- CLASS_STAR (for reference, already filtered) ---
    if 'CLASS_STAR_HSC_I' in tab_raw_final.colnames:
        class_star = np.array(tab_raw_final['CLASS_STAR_HSC_I'])
        class_star = np.where(
            (class_star >= 0) & (class_star <= 1),
            class_star, np.nan
        )
        data_dict['class_star'] = class_star
    
    # --- Other useful columns ---
    if 'OBJ_TYPE' in tab_raw_final.colnames:
        data_dict['obj_type'] = np.array(tab_raw_final['OBJ_TYPE'])
    
    if 'ZPHOT' in tab_raw_final.colnames:
        data_dict['photo_z'] = np.array(tab_raw_final['ZPHOT'])
    
    # Create DataFrame
    df = pd.DataFrame(data_dict)
    
    # --- Compute colors (will be NaN if either band is non-detection) ---
    for color_name, band1, band2 in COLORS_TO_COMPUTE:
        if band1 in df.columns and band2 in df.columns:
            df[color_name] = df[band1] - df[band2]
    
    if verbose:
        print(f"\n    Final DataFrame:  {df.shape[0]} rows, {df.shape[1]} columns")
        print(f"    is_lbg distribution: {df['is_lbg'].value_counts().to_dict()}")
        
        # Report non-detection rates
        print(f"\n    Non-detection rates (flag=0):")
        for band in bands:
            flag_col = f'flag_{band}'
            if flag_col in df.columns:
                non_det_rate = (df[flag_col] == 0).mean() * 100
                print(f"      {band}-band: {non_det_rate:.1f}% non-detected")
    
    return df


print("Main processing function defined.")

Main processing function defined.


In [11]:
# ## Cell 6: Process Both Fields

# Process each field
dfs = []

for config in CONFIGS:
    df_field = process_single_field(
        path_specz=config['path_specz'],
        path_raw=config['path_raw'],
        field_name=config['field_name'],
        match_radius=MATCH_RADIUS_ARCSEC,
        remove_stars=True,  # Remove CLASS_STAR > 0.8
        require_reliable_z=True,
        verbose=True
    )
    dfs.append(df_field)
    print()

# Combine all fields
df_train = pd.concat(dfs, ignore_index=True)

print("=" * 70)
print("COMBINED DATASET")
print("=" * 70)
print(f"Total samples: {len(df_train):,}")

Processing:  COSMOS

[1] Loading data...
    Specz catalog:  4,486 sources
    Raw catalog:   5,263,013 sources

[2] Coordinate matching...
    Matched (< 1.0"): 4,486 (100.0%)

[3] Applying field quality selection...
    Pass field selection: 4,421 (98.6%)

[4] Removing photometric stars (CLASS_STAR > 0.8)...
    Stars removed: 1,940 (43.9%)
    Galaxies kept: 2,481

[5] Determining LBG labels...
    VI reliable: 559 (22.5%)
    RR reliable: 1527 (61.5%)
    Total reliable: 1727 (69.6%)
    LBG (z > 2.0): 736
    Non-LBG (z <= 2.0): 991

    After reliability filter: 1,727
    LBG:  736
    Non-LBG: 991

[6] Extracting photometric features (no limiting mag fill)...

    Final DataFrame:  1727 rows, 41 columns
    is_lbg distribution: {0: 991, 1: 736}

    Non-detection rates (flag=0):
      u-band: 39.7% non-detected
      g-band: 0.3% non-detected
      r-band: 0.0% non-detected
      i-band: 0.5% non-detected
      z-band: 0.9% non-detected
      y-band: 4.6% non-detected

Processin

In [12]:
# Cell 7: Dataset Summary and Statistics

print("=" * 70)
print("TRAINING DATASET SUMMARY")
print("=" * 70)

# --- Class distribution ---
print("\n[1] Class Distribution:")
print("-" * 40)
n_lbg = (df_train['is_lbg'] == 1).sum()
n_non_lbg = (df_train['is_lbg'] == 0).sum()
print(f"    LBG (positive):     {n_lbg:,} ({n_lbg/len(df_train)*100:.1f}%)")
print(f"    Non-LBG (negative): {n_non_lbg:,} ({n_non_lbg/len(df_train)*100:.1f}%)")
print(f"    Class ratio:        {n_lbg/n_non_lbg:.2f}: 1")

# --- By field ---
print("\n[2] By Field:")
print("-" * 40)
for field in df_train['field'].unique():
    mask = df_train['field'] == field
    n_total = mask.sum()
    n_pos = (df_train. loc[mask, 'is_lbg'] == 1).sum()
    n_neg = (df_train.loc[mask, 'is_lbg'] == 0).sum()
    print(f"    {field}:")
    print(f"      Total: {n_total:,}, LBG: {n_pos: ,} ({n_pos/n_total*100:.1f}%), Non-LBG: {n_neg:,}")

# --- Detection rates ---
print("\n[3] Detection Rates (SNR >= 5):")
print("-" * 40)
bands = ['u', 'g', 'r', 'i', 'z', 'y']
for band in bands:
    flag_col = f'flag_{band}'
    if flag_col in df_train.columns:
        det_rate = df_train[flag_col].mean() * 100
        print(f"    {band}-band: {det_rate:.1f}% detected, {100-det_rate:.1f}% non-detected (NaN)")

# --- Color statistics (only for valid detections) ---
print("\n[4] Color Statistics (valid detections only):")
print("-" * 40)
color_cols = ['u_g', 'g_r', 'r_i', 'i_z', 'z_y']
for col in color_cols:
    if col in df_train.columns:
        valid_data = df_train[col].dropna()
        n_valid = len(valid_data)
        if n_valid > 0:
            mean_val = valid_data.mean()
            std_val = valid_data.std()
            print(f"    {col}: mean={mean_val:.3f}, std={std_val:.3f}, N_valid={n_valid} ({n_valid/len(df_train)*100:.1f}%)")

# --- LBG vs Non-LBG comparison ---
print("\n[5] Color Comparison (LBG vs Non-LBG, valid detections):")
print("-" * 40)
for col in ['u_g', 'g_r', 'r_i']: 
    if col in df_train.columns:
        lbg_data = df_train. loc[df_train['is_lbg']==1, col].dropna()
        non_lbg_data = df_train.loc[df_train['is_lbg']==0, col].dropna()
        if len(lbg_data) > 0 and len(non_lbg_data) > 0:
            diff = lbg_data. mean() - non_lbg_data.mean()
            sep = abs(diff) / (lbg_data.std() + non_lbg_data.std()) * 2
            print(f"    {col}: LBG={lbg_data.mean():.3f} (n={len(lbg_data)}), "
                  f"Non-LBG={non_lbg_data.mean():.3f} (n={len(non_lbg_data)}), "
                  f"diff={diff:.3f}, separation={sep:.3f}")

# --- Detection flag comparison ---
print("\n[6] Detection Flag Comparison (LBG vs Non-LBG):")
print("-" * 40)
for band in ['u', 'g']: 
    flag_col = f'flag_{band}'
    if flag_col in df_train.columns:
        det_lbg = df_train.loc[df_train['is_lbg']==1, flag_col].mean() * 100
        det_non_lbg = df_train.loc[df_train['is_lbg']==0, flag_col].mean() * 100
        print(f"    {band}-band detection: LBG={det_lbg:.1f}%, Non-LBG={det_non_lbg:.1f}%")

TRAINING DATASET SUMMARY

[1] Class Distribution:
----------------------------------------
    LBG (positive):     1,893 (54.1%)
    Non-LBG (negative): 1,603 (45.9%)
    Class ratio:        1.18: 1

[2] By Field:
----------------------------------------
    COSMOS:
      Total: 1,727, LBG:  736 (42.6%), Non-LBG: 991
    XMM-LSS:
      Total: 1,769, LBG:  1,157 (65.4%), Non-LBG: 612

[3] Detection Rates (SNR >= 5):
----------------------------------------
    u-band: 73.9% detected, 26.1% non-detected (NaN)
    g-band: 99.8% detected, 0.2% non-detected (NaN)
    r-band: 99.9% detected, 0.1% non-detected (NaN)
    i-band: 99.2% detected, 0.8% non-detected (NaN)
    z-band: 99.0% detected, 1.0% non-detected (NaN)
    y-band: 78.4% detected, 21.6% non-detected (NaN)

[4] Color Statistics (valid detections only):
----------------------------------------
    u_g: mean=1.173, std=0.543, N_valid=2583 (73.9%)
    g_r: mean=0.407, std=0.275, N_valid=3487 (99.7%)
    r_i: mean=0.123, std=0.194, 

In [14]:
# Cell 8: Data Validation

print("=" * 70)
print("DATA VALIDATION")
print("=" * 70)

# --- Check 1: No unreliable labels ---
print("\n[1] Label Validation:")
n_unreliable = (df_train['is_lbg'] == -1).sum()
print(f"    Unreliable labels (is_lbg == -1): {n_unreliable}")
assert n_unreliable == 0, f"ERROR: {n_unreliable} unreliable labels found!"
print("    ✓ All labels are reliable")

# --- Check 2: Star removal verification ---
print("\n[2] Star Removal Verification:")
if 'class_star' in df_train. columns:
    valid_cs = df_train['class_star'].dropna()
    n_stars = (valid_cs > CLASS_STAR_THRESHOLD).sum()
    print(f"    Sources with CLASS_STAR > {CLASS_STAR_THRESHOLD}: {n_stars}")
    if n_stars > 0:
        print(f"    ⚠️ WARNING: {n_stars} potential stars still present!")
    else:
        print("    ✓ All photometric stars removed")

# --- Check 3:  Redshift distribution ---
print("\n[3] Redshift Distribution:")
z = df_train['z_best']
print(f"    Min:  {z.min():.3f}")
print(f"    Max:  {z.max():.3f}")
print(f"    Mean: {z.mean():.3f}")
print(f"    Median: {z.median():.3f}")

# Check z ~ 0 sources
n_z_zero = (z < 0.1).sum()
print(f"\n    Sources with z < 0.1: {n_z_zero} ({n_z_zero/len(df_train)*100:.1f}%)")

# --- Check 4: Non-detection handling ---
print("\n[4] Non-detection Handling:")
for band in ['u', 'g', 'i']: 
    mag_col = f'mag_{band}'
    flag_col = f'flag_{band}'
    if mag_col in df_train. columns and flag_col in df_train.columns:
        n_nan = df_train[mag_col].isna().sum()
        n_flag_zero = (df_train[flag_col] == 0).sum()
        print(f"    {band}-band: {n_nan} NaN magnitudes, {n_flag_zero} flag=0")
        assert n_nan == n_flag_zero, f"Mismatch in {band}-band!"
print("    ✓ NaN magnitudes match non-detection flags")

print("\n" + "=" * 70)
print("✓ ALL VALIDATION CHECKS PASSED")
print("=" * 70)

DATA VALIDATION

[1] Label Validation:
    Unreliable labels (is_lbg == -1): 0
    ✓ All labels are reliable

[2] Star Removal Verification:
    Sources with CLASS_STAR > 0.8: 0
    ✓ All photometric stars removed

[3] Redshift Distribution:
    Min:  0.000
    Max:  5.287
    Mean: 1.902
    Median: 2.444

    Sources with z < 0.1: 588 (16.8%)

[4] Non-detection Handling:
    u-band: 912 NaN magnitudes, 912 flag=0
    g-band: 6 NaN magnitudes, 6 flag=0
    i-band: 29 NaN magnitudes, 29 flag=0
    ✓ NaN magnitudes match non-detection flags

✓ ALL VALIDATION CHECKS PASSED


In [15]:
# ## Cell 9: Preview Sample Data

print("=" * 70)
print("SAMPLE DATA PREVIEW")
print("=" * 70)

# Select columns to display
preview_cols = [
    'field', 'z_best', 'is_lbg',
    'mag_u', 'mag_g', 'mag_i',
    'u_g', 'g_r', 'r_i',
    'flag_u', 'flag_g', 'class_star'
]
preview_cols = [c for c in preview_cols if c in df_train.columns]

print("\nFirst 10 rows:")
print(df_train[preview_cols].head(10).to_string())

print("\n\nLBG samples with u-band detection (first 5):")
lbg_u_det = df_train[(df_train['is_lbg']==1) & (df_train['flag_u']==1)]
print(lbg_u_det[preview_cols].head().to_string())

print("\n\nLBG samples with u-band NON-detection (first 5):")
lbg_u_nondet = df_train[(df_train['is_lbg']==1) & (df_train['flag_u']==0)]
print(lbg_u_nondet[preview_cols]. head().to_string())

print("\n\nNon-LBG samples with z < 0.1 (first 5):")
low_z = df_train[(df_train['is_lbg']==0) & (df_train['z_best']<0.1)]
print(low_z[preview_cols].head().to_string())

SAMPLE DATA PREVIEW

First 10 rows:
    field    z_best  is_lbg      mag_u      mag_g      mag_i       u_g       g_r       r_i  flag_u  flag_g  class_star
0  COSMOS  0.000848       0  25.314104  24.497484  24.231743  0.816620  0.185917  0.079824       1       1    0.000160
1  COSMOS  0.000848       0        NaN  24.152698  23.788873       NaN  0.147650  0.216175       0       1    0.009659
2  COSMOS  1.014920       0  25.312200  24.432407  23.930747  0.879793  0.260422  0.241238       1       1    0.056854
3  COSMOS  0.000848       0  25.946995  25.030402  24.239579  0.916594  0.549349  0.241474       1       1    0.001913
4  COSMOS  3.018500       1        NaN  24.958986  23.980751       NaN  0.717745  0.260490       0       1    0.066816
5  COSMOS  0.000848       0  25.393914  24.466287  24.341165  0.927628  0.229330 -0.104208       1       1    0.619873
6  COSMOS  0.000848       0  25.541424  24.836403  24.333199  0.705021  0.401859  0.101345       1       1    0.733707
7  COSMOS  0

In [16]:
# ## Cell 10: Save Dataset

# Define output path
output_dir = "../data_processed"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path. join(output_dir, "training_dataset_v2.csv")

# Select columns to save
core_cols = ['field', 'specz_idx', 'raw_idx', 'RA', 'DEC', 'z_best', 'is_lbg', 'z_source']

feature_cols = []
for band in ['u', 'g', 'r', 'i', 'z', 'y']: 
    feature_cols.extend([f'mag_{band}', f'err_{band}', f'snr_{band}', f'flag_{band}'])

color_cols = ['u_g', 'g_r', 'r_i', 'i_z', 'z_y']
extra_cols = ['offset_mag', 'class_star', 'obj_type', 'photo_z']

all_cols = core_cols + feature_cols + color_cols + extra_cols
save_cols = [c for c in all_cols if c in df_train.columns]

# Save
df_save = df_train[save_cols].copy()
df_save.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")
print(f"Shape: {df_save.shape}")
print(f"Columns ({len(save_cols)}): {save_cols}")

Dataset saved to: ../data_processed/training_dataset_v2.csv
Shape: (3496, 41)
Columns (41): ['field', 'specz_idx', 'raw_idx', 'RA', 'DEC', 'z_best', 'is_lbg', 'z_source', 'mag_u', 'err_u', 'snr_u', 'flag_u', 'mag_g', 'err_g', 'snr_g', 'flag_g', 'mag_r', 'err_r', 'snr_r', 'flag_r', 'mag_i', 'err_i', 'snr_i', 'flag_i', 'mag_z', 'err_z', 'snr_z', 'flag_z', 'mag_y', 'err_y', 'snr_y', 'flag_y', 'u_g', 'g_r', 'r_i', 'i_z', 'z_y', 'offset_mag', 'class_star', 'obj_type', 'photo_z']


In [18]:
# ## Cell 11: Final Summary

print("=" * 70)
print("FINAL SUMMARY - Training Dataset V2")
print("=" * 70)
print(f"""
Key Changes from V1:
  ✓ Removed photometric stars (CLASS_STAR > {CLASS_STAR_THRESHOLD})
  ✓ No limiting magnitude filling (non-detections are NaN)
  ✓ Detection flags preserved for model training

Dataset Statistics:
  Total samples:     {len(df_train):,}
  LBG (is_lbg=1):   {(df_train['is_lbg']==1).sum():,} ({(df_train['is_lbg']==1).mean()*100:.1f}%)
  Non-LBG (is_lbg=0): {(df_train['is_lbg']==0).sum():,} ({(df_train['is_lbg']==0).mean()*100:.1f}%)
  
  By Field:
    COSMOS:  {(df_train['field']=='COSMOS').sum():,}
    XMM-LSS: {(df_train['field']=='XMM-LSS').sum():,}

Detection Rates:
  u-band: {df_train['flag_u'].mean()*100:.1f}% detected
  g-band: {df_train['flag_g'].mean()*100:.1f}% detected
  i-band: {df_train['flag_i'].mean()*100:.1f}% detected

Output File: {output_path}
""")

FINAL SUMMARY - Training Dataset V2

Key Changes from V1:
  ✓ Removed photometric stars (CLASS_STAR > 0.8)
  ✓ No limiting magnitude filling (non-detections are NaN)
  ✓ Detection flags preserved for model training

Dataset Statistics:
  Total samples:     3,496
  LBG (is_lbg=1):   1,893 (54.1%)
  Non-LBG (is_lbg=0): 1,603 (45.9%)
  
  By Field:
    COSMOS:  1,727
    XMM-LSS: 1,769

Detection Rates:
  u-band: 73.9% detected
  g-band: 99.8% detected
  i-band: 99.2% detected

Output File: ../data_processed/training_dataset_v2.csv

